# sagekit offload: a faction's full-quality builds on an A100

Runtime: **A100 GPU** (Runtime > Change runtime type). The steps are in [docs/OFFLOAD.md](../../docs/OFFLOAD.md).

1. On the Mac: `python3 -m sagekit offload pack <faction>`, then put `build/offload/<faction>-<date>.zip` in your
   Google Drive (folder `sagekit/`).
2. Here: set `BUNDLE` below, then Runtime > Run all. Nothing here reads or needs the game.
3. The results zip lands next to the bundle in Drive. On the Mac: `python3 -m sagekit offload unpack <results.zip>`.

The bundle holds EA's files (the game snapshot and each building's `src/`): keep it in your private Drive,
never share the folder, and delete both zips when the faction is installed.

In [ ]:
BUNDLE = "/content/drive/MyDrive/sagekit/men-20260928.zip"   # the zip `offload pack` wrote
BUILDS = 5            # buildings at once
SLOTS = 5             # Blender processes at once (SAGEKIT_BLENDER_SLOTS), 4-6 on an A100
ONLY = ""             # "" for all, or "well,statue"
KEEP_BAKES = False    # True: work/bake/*.npy come back too (about 0.5 GB per building)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Tools: Blender 4.5.9 (the Mac's version), the OpenSAGE W3D add-on v0.7.2 (the Mac's copy), ImageMagick

In [ ]:
%%bash
set -e
cd /content
apt-get -qq update >/dev/null
apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 libegl1 \
    imagemagick fonts-dejavu-core unzip >/dev/null
# Blender, the official Linux build, checked
if [ ! -x blender-4.5.9-linux-x64/blender ]; then
  wget -q https://download.blender.org/release/Blender4.5/blender-4.5.9-linux-x64.tar.xz
  echo "dcdc3eca6c9825bb35a8033b689c053f3cb5a9b0cd2a61b2eac2a49436b4ad3d  blender-4.5.9-linux-x64.tar.xz" | sha256sum -c
  tar xf blender-4.5.9-linux-x64.tar.xz && rm blender-4.5.9-linux-x64.tar.xz
fi
# the W3D add-on: the release byte-identical to the Mac's install, checked, then enabled
ADDONS=~/.config/blender/4.5/scripts/addons
if [ ! -d $ADDONS/io_mesh_w3d ]; then
  wget -q -O io_mesh_w3d.zip https://github.com/OpenSAGE/OpenSAGE.BlenderPlugin/releases/download/v0.7.2/io_mesh_w3d.zip
  echo "4c07e00977ad4ae9273c57107187a8bbe0fe865a40ac972584c4ff43ba43cd94  io_mesh_w3d.zip" | sha256sum -c
  mkdir -p $ADDONS && unzip -q -o io_mesh_w3d.zip -d $ADDONS
fi
blender-4.5.9-linux-x64/blender -b --python-expr \
  "import bpy; bpy.ops.preferences.addon_enable(module='io_mesh_w3d'); bpy.ops.wm.save_userpref()" | tail -1
# ImageMagick 7's `magick` on Ubuntu's ImageMagick 6, without its small resource caps
printf '#!/bin/sh\nif [ "$1" = identify ]; then shift; exec identify "$@"; fi\nexec convert "$@"\n' > /usr/local/bin/magick
chmod +x /usr/local/bin/magick
sed -i '/name="\(width\|height\|area\|memory\|map\|disk\)"/d' /etc/ImageMagick-6/policy.xml
magick -version | head -1

### Unpack the bundle (unzip keeps the files' dates: the recolour caches rely on them)

In [ ]:
import os
!rm -rf /content/sagekit && mkdir -p /content/sagekit
!cp "$BUNDLE" /content/bundle.zip && cd /content/sagekit && unzip -q /content/bundle.zip && rm /content/bundle.zip
os.chdir("/content/sagekit")
os.environ.update(SAGEKIT_BLENDER="/content/blender-4.5.9-linux-x64/blender", SAGEKIT_BLENDER_SLOTS=str(SLOTS),
                  SAGEKIT_GAME_SNAPSHOT="/content/sagekit/_offload/game", PYTHONPATH="/content/sagekit")
!cat _offload/manifest.json

In [ ]:
# the GPU Cycles will use (expect: cycles on OPTIX: NVIDIA A100...)
!$SAGEKIT_BLENDER -b --python-expr "import sys; sys.path.insert(0, '.'); import bpy; from sagekit.blender import scene; scene.use_gpu(bpy.context.scene)" | grep -i "cycles\|gpu"

### Build (geometry to render for every building; a timing table at the end)

In [ ]:
!python3 -m sagekit offload run --builds $BUILDS {"--only " + ONLY if ONLY else ""}

### Results back to Drive

In [ ]:
import glob, shutil
!python3 -m sagekit offload results {"--with-bakes" if KEEP_BAKES else ""}
for z in glob.glob("build/offload/*-results.zip"):
    shutil.copy(z, os.path.dirname(BUNDLE))
    print("in Drive:", os.path.join(os.path.dirname(BUNDLE), os.path.basename(z)))